# Plant Disease Dataset EDA
This notebook inspects class distribution and sample images from the prepared PlantVillage split.

In [ ]:
from pathlib import Path
from collections import defaultdict
import random
import matplotlib.pyplot as plt

project_root = Path('..').resolve()
data_root = project_root / 'data'
print('Data root:', data_root)

In [ ]:
def count_images(split_path):
    stats = {}
    if not split_path.exists():
        return stats
    for class_dir in sorted([d for d in split_path.iterdir() if d.is_dir()]):
        count = len([p for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
        stats[class_dir.name] = count
    return stats

train_stats = count_images(data_root / 'train')
val_stats = count_images(data_root / 'val')
test_stats = count_images(data_root / 'test')

print('Train classes:', len(train_stats))
print('Train samples:', sum(train_stats.values()))
print('Val samples:', sum(val_stats.values()))
print('Test samples:', sum(test_stats.values()))

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'class_name': list(train_stats.keys()),
    'train_count': [train_stats[k] for k in train_stats.keys()],
    'val_count': [val_stats.get(k, 0) for k in train_stats.keys()],
    'test_count': [test_stats.get(k, 0) for k in train_stats.keys()]
})
df.sort_values('train_count', ascending=False).head(20)

In [ ]:
# Show random images from train split
train_root = data_root / 'train'
class_dirs = [d for d in train_root.iterdir() if d.is_dir()] if train_root.exists() else []
sample_paths = []
for class_dir in class_dirs[:6]:
    imgs = [p for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    if imgs:
        sample_paths.append(random.choice(imgs))

cols = 3
rows = max(1, (len(sample_paths) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, ax in enumerate(axes):
    ax.axis('off')
    if i < len(sample_paths):
        img_path = sample_paths[i]
        img = plt.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.parent.name)

plt.tight_layout()
plt.show()